# Lab 13: How to give an agent a search tool

**Notebook 2 of 2: the search tool at the Chinook support desk**

## Goal

Give the model a search over the support team's files as a tool, so that it answers a support rep's questions from
those files. Chinook is a sample music shop; its support team keeps 20 policy files and 16 ticket files. A customer
whose warped vinyl was replaced once writes back: the replacement arrived warped too. The rep asks what to offer now
and how long the refund takes. The rule for a second damaged copy is in a policy that only the old ticket names. One
search with the rep's question misses that rule. Given the search as a tool, the model reads the ticket first and
writes its next query in the ticket's words. Then we compare it with a single search that simply returns more files:
cheaper on this wording, and not enough when the rep asks the same thing in other words.

## What you will learn

- How to build a search over the shop's 36 files with embeddings and NumPy, and read which files one search returns.
- How to give the support model the search as a tool, and read every query it writes, turn by turn.
- How to tell a query written from the question from a query written from a file the model has just read.
- How to compare the agent with a single search that returns more files (a bigger `k`), in model calls and tokens,
  and say when the bigger `k` is enough.
- How a rewording of the same question moves a policy down the ranking, and what that does to each approach.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model names from `models.env`,
and takes the key from `.env` without printing it.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


Next, the client (60 seconds per request, so a stuck request fails quickly) and the embedding model `models.env`
names. Every chat call goes to Chat Completions and passes `reasoning_effort=EFFORT or omit`, which leaves the argument
out for a model that takes no effort setting.

In [2]:
import json
import numpy as np
from openai import OpenAI, omit

client = OpenAI(timeout=60)
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]
print("embedding model:", EMBEDDING_MODEL)

embedding model: text-embedding-3-small


---

## Part A: State the case, and see one search miss the rule for a second damaged copy

This part sets out the rep's question and what a right answer contains, builds the search, and runs it once with the
question's words.

### The shop's files and ticket 06

The files are in `data/corpus/` at the repository folder, one Markdown file each. We read them in place and never
change them. `FILES` maps each file name to its text. Then the ticket this case is about.

In [3]:
CORPUS = ROOT / "data" / "corpus"
FILES = {path.name: path.read_text() for path in sorted(CORPUS.glob("*.md"))}
print(len(FILES), "files")
print()
print(FILES["ticket_06_germany_warped_vinyl.md"])

36 files

# Ticket 06 — Germany, warped vinyl

Date: 2012-07-18
Customer: billing country Germany

Vinyl arrived warped. Customer sent a photo. Damaged-media policy: replace once at no charge.

Status: resolved. Replacement shipped 2012-07-20.



36 files: 20 policies and 16 tickets, each name starting with `policy_` or `ticket_`. Ticket 06 is a record of a case that is closed: in July 2012 a customer in Germany received a vinyl record
that arrived warped and sent a photo, and the shop sent one replacement at no charge under its damaged-media policy.
The replacement shipped on 20 July. The ticket names the policy ("Damaged-media policy: replace once at no charge")
and does not say what the policy does when a copy arrives damaged a second time.

### The rep's question, and what a right answer contains

Now the same customer writes back: the replacement arrived warped too. The rep asks:

> *Ticket 06's replacement arrived warped too. What do we offer now, and how long does the refund take?*

A right answer has two facts, each in its own policy file:

1. **What to offer now:** a refund instead of a third copy. `policy_damaged_media.md`: "If the same title arrives
   damaged twice, offer a refund instead of a third copy."
2. **How long the refund takes:** 5 to 10 business days after support approves the request, back to the original
   payment method. `policy_refunds.md`. Fact 2 is this time, from the approval to the refund. The time support takes
   to decide is a different number: `policy_support_hours.md` puts the decision "within 3 business days of the first
   reply".

Here are the two policies, so you can check every answer in this notebook against them.

In [4]:
print(FILES["policy_damaged_media.md"])
print(FILES["policy_refunds.md"])

# Damaged media

If a CD or vinyl arrives cracked, warped, or will not play, we replace it at no charge. Photograph the damage. Keep the original copy until the replacement arrives, then recycle it.

If the same title arrives damaged twice, offer a refund instead of a third copy.

# Refunds

Refunds go back to the original payment method. They take 5 to 10 business days after support approves the request.

Partial refunds are allowed when only some tracks on a multi-disc set are defective.

We do not refund because a customer changed their mind about a digital purchase.

Duplicate charges: if two invoices exist for the same album on the same day, refund the later invoice in full. Attach both invoice numbers.



The question names neither policy. It names ticket 06, and the ticket names the damaged-media policy. So the way to
fact 1 goes through the ticket: read the ticket, then look up the policy it names.

### Build the search

The search works as for any set of short texts. An embedding model turns each file into a vector of 1,536 numbers, so
that texts with a similar meaning get vectors that point in similar directions. We embed all 36 files in one request
and stack the vectors as the rows of `D`. To search, we embed the query into one vector `q` and compute `D @ q`, one
score per file: OpenAI's embedding vectors have length 1, so each score is the cosine similarity of the query and a
file, and the closer to 1, the closer in meaning. A NumPy array is the whole search here; a vector database such as
Chroma does the same job for far more documents.

In [5]:
names = list(FILES)
response = client.embeddings.create(model=EMBEDDING_MODEL, input=list(FILES.values()))

# Row i of D is the vector of the file names[i]
D = np.array([item.embedding for item in response.data])
print(D.shape)

(36, 1536)


`(36, 1536)`: 36 rows, one vector per file in the order of `names`, and 1,536 numbers in each.

`nearest` returns the names of the `k` nearest files with their scores, best first; `np.argsort(-scores)` lists the
positions from the highest score to the lowest. We print the six files nearest to the rep's question.

In [6]:
def nearest(query, k):
    """The k files nearest to the query, as (file name, score) pairs, nearest first."""
    q = np.array(client.embeddings.create(model=EMBEDDING_MODEL, input=query).data[0].embedding)
    scores = D @ q
    return [(names[i], float(scores[i])) for i in np.argsort(-scores)[:k]]


QUESTION = "Ticket 06's replacement arrived warped too. What do we offer now, and how long does the refund take?"

for rank, (name, score) in enumerate(nearest(QUESTION, 6), start=1):
    print(f"{rank}. {score:.3f}  {name}")

1. 0.603  ticket_06_germany_warped_vinyl.md
2. 0.494  ticket_01_helena_wrong_album.md
3. 0.482  policy_refunds.md
4. 0.482  ticket_07_uk_preorder_delay.md
5. 0.453  policy_damaged_media.md
6. 0.448  ticket_11_ireland_exchange.md


Reading the six from the top:

- **1st, ticket 06** (0.603): the question names it.
- **2nd, ticket 01** (0.494): Helena's wrong album, another case about a replacement.
- **3rd, `policy_refunds.md`** (0.482): fact 2, the refund time.
- **4th, ticket 07** (0.482): a cancelled preorder refunded in 8 business days, one customer's case, not the rule.
  It ties with the refund policy to three decimals.
- **5th, `policy_damaged_media.md`** (0.453): fact 1. The question says "warped" and "replacement"; the rule says
  "the same title arrives damaged twice" and "a third copy".
- **6th, ticket 11** (0.448): an exchange of a CD for a vinyl.

A search that returns the 2 nearest files gets neither policy.

### Answer from one search

`search` returns two things: the names of the nearest files, for us to print, and the text the model reads, each
file's name in brackets followed by its text.

`answer_once` is **RAG**, retrieval-augmented generation, in its simplest form: one search with the question's words,
the files it found and the question in one request, and one answer. We call that **single-shot retrieval**.
`answer_once` returns the whole response, so we can count its tokens later. We run it with `k=2`, the number of files
each search returns in the rest of this notebook.

In [7]:
def search(query, k=2):
    """The names of the k nearest files, and one text with each file's name in brackets, then its text."""
    found = [name for name, _ in nearest(query, k)]
    return found, "\n\n".join(f"[{name}]\n{FILES[name]}" for name in found)


ANSWER_RULES = "Answer only from the files. If they do not say, say you do not know."


def answer_once(question, k):
    files, text = search(question, k)
    return client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": ANSWER_RULES},
            {"role": "user", "content": f"Files:\n\n{text}\n\nQuestion: {question}"},
        ],
        reasoning_effort=EFFORT or omit,
    )


single_k2 = answer_once(QUESTION, k=2)
print(single_k2.choices[0].message.content)

Ticket 06 says damaged media is replaced once at no charge, and that replacement has already shipped. The files don’t say what to offer if the replacement is also warped or how long a refund takes, so I don’t know.


The reply: *“Ticket 06 says damaged media is replaced once at no charge, and that replacement has already shipped.
The files don’t say what to offer if the replacement is also warped or how long a refund takes, so I don’t know.”*
The model had ticket 06 and ticket 01, kept to the rule to answer only from them, and did not invent a rule. **The
miss is in the search, not in the answer.**

**Checkpoint A.** 36 files. Of the six files nearest the question, `policy_refunds.md` is 3rd and
`policy_damaged_media.md` 5th. With `k=2` the answer has neither fact and says the files do not say.

---

## Part B: Give the model the search as a tool, and read the query it writes after the ticket

A person would read ticket 06, see "Damaged-media policy", and look that policy up. For the model to do the same, it
must write its own queries. This part gives it the search as a tool and runs the first two turns by hand.

### The tool and the instructions

The tool schema describes `search` to the model: its name, what it does, and one string argument, `query`.
`strict: True` makes the model's arguments match the schema exactly; strict mode needs every argument in
`required` and `additionalProperties` set to `False`. The model never sees our Python; it sees this description,
and our code runs the search when the model asks.

The system message has four rules: answer only from the shop's files, use the search tool, search again while the
question is still open, and say "I do not know" when the files do not say. The third rule matters: without it, a model
can write its answer after one search even when that search returned only a pointer to another document (notebook 1,
Part B, shows this on a smaller case). It says nothing about damaged media, refunds or tickets.

We also pass `parallel_tool_calls=False`, which asks for at most one tool call per reply. Without it, the model sent
two searches at once on turn 1 in most of our runs, one for each part of the question, and in a few of those the split
already found the damaged-media policy, so there was no query written from the ticket to read. With one search per
turn, each turn shows one query and the files it found.

In [8]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the shop's policy and ticket files with a short query of a few words. Returns the two nearest files.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

SYSTEM = (
    "Answer only from the shop's files. Use the search tool to find them. "
    "Search again while the question is still open. If the files do not say, say you do not know."
)

### Turn 1: the first query

A **turn** is one call to the model. We send the system message, the question and the tool, and print the reply's
`finish_reason` (why the model stopped writing) and the query the model writes. The arguments arrive as a string of
JSON; `json.loads` turns them into a dictionary.

In [9]:
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": QUESTION},
]

response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=[search_tool], parallel_tool_calls=False, reasoning_effort=EFFORT or omit
)
reply = response.choices[0].message
call = reply.tool_calls[0]
query_1 = json.loads(call.function.arguments)["query"]
print("finish_reason:", response.choices[0].finish_reason)
print("query 1:", query_1)

finish_reason: tool_calls
query 1: Ticket 06 replacement warped refund time offer


`finish_reason` is `tool_calls`: the model is asking for the tool, not answering. The first query is *Ticket 06
replacement warped refund time offer*: every word but *time* is in the question.

Our code runs the search and sends the result back: the model's reply goes into `messages`, then a `tool` message with
the two files' text, whose `tool_call_id` names the call it answers. We print only the names of the files the search
returned; the model gets their whole text.

In [10]:
files, result = search(query_1)
messages.append(reply)
messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
print("files returned:", files)

files returned: ['ticket_06_germany_warped_vinyl.md', 'ticket_07_uk_preorder_delay.md']


The search returned ticket 06 and ticket 07. The ticket's text, with "Damaged-media policy: replace once at no charge", is now in `messages`.

### Turn 2: the query written after the ticket

The same call with the longer list. Before you run it: what will the model search for next?

In [11]:
response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=[search_tool], parallel_tool_calls=False, reasoning_effort=EFFORT or omit
)
reply = response.choices[0].message
call = reply.tool_calls[0]
query_2 = json.loads(call.function.arguments)["query"]
files, result = search(query_2)
print("finish_reason:", response.choices[0].finish_reason)
print("query 2:", query_2)
print("files returned:", files)

finish_reason: tool_calls
query 2: replacement arrived damaged warped second replacement refund Germany policy
files returned: ['ticket_06_germany_warped_vinyl.md', 'policy_damaged_media.md']


`finish_reason` is `tool_calls` again. The second query is *replacement arrived damaged warped second replacement refund Germany policy*, and this search returned `policy_damaged_media.md`, the file with fact 1.

### The two queries side by side

| Query | Where its words come from |
|---|---|
| 1: *Ticket 06 replacement warped refund time offer* | the question, plus *time*, the model's own word |
| 2: *replacement arrived damaged warped second replacement refund Germany policy* | the question (*replacement*, *arrived*, *warped*, *refund*); ticket 06 (*damaged* and *policy*, from "Damaged-media policy", and *Germany*, from "billing country Germany"); and *second*, the model's own word |

The question never says *damaged*, *policy* or *Germany*; ticket 06 says all three. *Damaged* is the word the query
now shares with the policy: its title is "Damaged media", and its rule says "arrives damaged twice". A query written
from a file the model has just read is what a single search can never write: its only query is the question, fixed
before anything is read. This is **agentic RAG**: retrieval given to the model as a tool, so that it writes the
queries and searches again on what it found.

**Checkpoint B.** Turn 1: one query built from the question's words, returning ticket 06. Turn 2: a query with words from ticket 06 that are not in the question (ours: *damaged*, *policy* and *Germany*), returning `policy_damaged_media.md`.

---

## Part C: Run the search in a loop until the model answers both parts

The turns from here on repeat the same two moves: call the model, and run the search it asks for. `run_loop` does
that from a fresh start, on each turn:

1. Send `messages` with the tool.
2. Add the reply to `messages`.
3. If the reply has no tool call, it is the answer: return.
4. Otherwise run `search` for the tool call, print the query and the files, and add the `tool` message.

At most `max_turns` turns: that is the **turn cap**, and a run it stops returns `None` as its answer. Besides the
answer, `run_loop` returns `queries`, one `(turn, query, files)` entry per search, and `usage`, the token counts of
every model call, which Part D compares with a single search.

In [12]:
def run_loop(question, max_turns=6):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    queries, usage = [], []

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=[search_tool], parallel_tool_calls=False, reasoning_effort=EFFORT or omit
        )
        usage.append(response.usage)
        reply = response.choices[0].message
        messages.append(reply)

        # The model answered: return its text, the queries and the token counts
        if not reply.tool_calls:
            return reply.content, queries, usage

        for call in reply.tool_calls:
            query = json.loads(call.function.arguments)["query"]
            files, result = search(query)
            print(f"turn {turn}: search({query!r}) -> {files}")
            queries.append((turn, query, files))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    # The turn cap stopped the loop: no answer
    return None, queries, usage

In [13]:
answer, queries, usage = run_loop(QUESTION)
print()
print(answer)

turn 1: search('Ticket 06 replacement warped refund options processing time') -> ['ticket_06_germany_warped_vinyl.md', 'policy_refunds.md']


turn 2: search('damaged media replacement warped again second replacement refund offer') -> ['policy_damaged_media.md', 'ticket_06_germany_warped_vinyl.md']



Since the same title arrived warped twice, offer a **refund instead of a third copy**. Refunds go to the original payment method and take **5–10 business days after support approves the request**.


Two searches and then the answer. The loop starts afresh, so its queries differ from Part B's. Turn 1 searched
*Ticket 06 replacement warped refund options processing time* and found ticket 06 and `policy_refunds.md`, so fact 2
was already in. Turn 2 searched *damaged media replacement warped again second replacement refund offer*, with
*damaged media* from the ticket, and found `policy_damaged_media.md`. The third model call wrote the answer.

Check the answer against the two policies printed in Part A: a refund instead of a third copy (fact 1), and 5 to 10 business days after support approves it, back to the original payment method (fact 2). Both are there, in the words of their policies.

**Checkpoint C.** At least two searches: one that returns ticket 06, and a later one with words from the ticket that returns `policy_damaged_media.md`; `policy_refunds.md` comes back from one of them. The answer has both facts. Ours: two searches, *damaged media* in the second, three model calls.

The queries change from run to run. Most runs end with both facts. In a few, the search for the refund time
returns only tickets, and the answer says it could not find the refund time; if yours does that, read the queries
and run the cell again.

---

## Part D: Find when a bigger `k` is enough, and when the agent finds a policy the bigger `k` misses

The agent made 3 model calls. A simpler fix is to give the single search more files: in Part A the damaged-media
policy was 5th nearest, so `k=6`, the six files printed there, reaches it with one file to spare. This part tries
that, then asks the same question in other words.

### A single search with `k=6`

In [14]:
single_k6 = answer_once(QUESTION, k=6)
print(single_k6.choices[0].message.content)
print()
print("model calls: 1 | input tokens:", single_k6.usage.prompt_tokens)
print("agent, Part C: model calls:", len(usage), "| input tokens:", sum(u.prompt_tokens for u in usage))

Offer a refund rather than a third copy, since the same title arrived damaged twice. The refund goes to the original payment method and takes 5–10 business days after support approves it.

model calls: 1 | input tokens: 555
agent, Part C: model calls: 3 | input tokens: 1196


Both facts in one call: *“Offer a refund rather than a third copy, since the same title arrived damaged twice. The
refund goes to the original payment method and takes 5–10 business days after support approves it.”* One call with
555 input tokens, against the agent's 3 calls with 1,196: more than twice the input for the same two facts.

Why does the agent send so many more tokens? Every turn sends the whole message list again: the system message, the
question, the tool's description and every file read so far. On this wording a bigger `k` is the cheaper fix.

### The same question in other words

A rep could ask the same thing differently. Here is a rewording that says "money back" where the first said "refund",
and the rank of each policy among all 36 files for both wordings.

In [15]:
REWORDED = "The replacement for ticket 06 came warped as well. What now, and when does the customer get the money back?"

for question in (QUESTION, REWORDED):
    ranking = [name for name, _ in nearest(question, 36)]
    print(question)
    for name in ("policy_damaged_media.md", "policy_refunds.md"):
        print(f"   {name:25} rank {ranking.index(name) + 1} of 36")

Ticket 06's replacement arrived warped too. What do we offer now, and how long does the refund take?
   policy_damaged_media.md   rank 5 of 36
   policy_refunds.md         rank 3 of 36


The replacement for ticket 06 came warped as well. What now, and when does the customer get the money back?
   policy_damaged_media.md   rank 6 of 36
   policy_refunds.md         rank 10 of 36


For the rep's wording the two policies are 5th and 3rd, both inside `k=6`. For the rewording,
`policy_damaged_media.md` is 6th, still just inside, and `policy_refunds.md` falls to 10th: the rewording asks when the
customer gets "the money back", and the policy talks about refunds and business days.

Single-shot with `k=6` on the rewording:

In [16]:
single_k6_reworded = answer_once(REWORDED, k=6)
print(single_k6_reworded.choices[0].message.content)

The damaged-media policy says that if the same title arrives damaged twice, offer a refund instead of sending a third copy. It doesn’t say when the customer would receive the money back.


Fact 1 and no fact 2: *“... It doesn’t say when the customer would receive the money back.”* The refund policy is not among the six files, and the model says so.

And the agent on the rewording:

In [17]:
answer_reworded, queries_reworded, usage_reworded = run_loop(REWORDED)
print()
print(answer_reworded)

turn 1: search('ticket 06 replacement warped refund money back') -> ['ticket_06_germany_warped_vinyl.md', 'policy_refunds.md']


turn 2: search('replacement still warped second replacement damaged media policy') -> ['policy_damaged_media.md', 'ticket_06_germany_warped_vinyl.md']


turn 3: search('refund processing Germany payment method timing approved ticket 06') -> ['ticket_06_germany_warped_vinyl.md', 'ticket_07_uk_preorder_delay.md']



Because the same title arrived damaged twice, the damaged-media policy says to offer a refund instead of sending a third copy. Refunds go to the original payment method and take **5–10 business days after support approves the request**. The files don’t say whether support has approved this refund yet, so I can’t give a specific date.


Both facts, in 4 model calls. Turn 1 searched *ticket 06 replacement warped refund money back*: *refund* is
the model's word, not the rewording's, and it brought `policy_refunds.md` at once. Turn 2 searched *replacement still
warped second replacement damaged media policy*, with *damaged media policy* from the ticket, and found the
damaged-media policy. Turn 3 searched for the refund timing again and found two tickets; the answer takes the refund
time from the policy turn 1 found.

**Lesson to Remember.** A bigger `k` is enough when the files an answer needs are near the question's words, and you
have checked their ranks on questions worded the way your users word them. There it is the cheaper choice by a wide
margin. The search tool is worth its extra calls when the question's own words do not reach a file the answer needs,
and the model can write a query whose words do: words from a file it has read, as Mia Rana's name did in notebook 1,
or words of its own, as *refund* did on the rewording here.

| Question | Single search, `k=6` | Agent |
|---|---|---|
| *Ticket 06's replacement arrived warped too ... how long does the refund take?* | both facts, 1 call, 555 input tokens | both facts, 3 calls, 1,196 input tokens |
| *The replacement for ticket 06 came warped as well ... money back?* | the refund time missing: `policy_refunds.md` is 10th | both facts, 4 calls |

Notice that the two questions are the same question to a person. Nothing about the search changed between the rows;
only the words did.

**Checkpoint D.** On the rep's question `k=6` gives both facts in one call (ours: 555 input tokens, against the agent's 1,196 over 3 calls). The rewording moves `policy_refunds.md` to rank 10; `k=6` then misses the refund time, and the agent gives both facts.

---

## What you can do now

- Build a search over the shop's files with one embeddings request and `D @ q`, and read a ranking (Part A).
- Say why a single search with the question's words misses a policy that only a ticket names (Part A).
- Give the model the search as a tool and read every query it writes, turn by turn (Part B).
- Tell a query written from the question from a query written from a file the model has just read (Part B).
- Run the searches in a loop with a turn cap and check the answer against the files (Part C).
- Compare the agent with a single search at a bigger `k`, in calls and tokens, and say when the bigger `k` is enough
  and when a rewording breaks it (Part D).

## Going further

These cells are optional. They use only what the notebook has already run, and make no new model calls.

### Which words of each query came from a file already read

`borrowed_words` does in code what we did by eye in Parts B and C, here on the queries that Part C's `run_loop` kept:
the words of a query that are not in the question, and of those, the ones that appear in a file returned by an earlier
search. `re.findall(r"[a-z0-9]+", ...)` splits a text into words; `set` keeps each word once; `-` keeps the words of
the first set that are not in the second, `&` the words in both.

In [18]:
import re


def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def borrowed_words(query, question, files_read):
    text_read = " ".join(FILES[name] for name in files_read)
    return sorted((words(query) - words(question)) & words(text_read))


for turn, query, files in queries:
    # the files every earlier turn returned
    read_before = [name for t, _, found in queries if t < turn for name in found]
    print(f"turn {turn}: {query!r:50} borrowed: {borrowed_words(query, QUESTION, read_before)}")

turn 1: 'Ticket 06 replacement warped refund options processing time' borrowed: []
turn 2: 'damaged media replacement warped again second replacement refund offer' borrowed: ['damaged', 'media']


Turn 2 borrows *damaged* and *media* from ticket 06 ("Damaged-media policy"), as we read by eye in Part C; turn 1
borrows nothing. The check finds words, not reasons. A word the model chose for itself can happen to be in a file it
read, and a query reworded entirely in the model's own words shows `[]` even when the idea came from a file. Read the
queries too.

### What each approach cost

The price of a call is its input tokens times the input price plus its output tokens times the output price, both from
`models.env` in US dollars per million tokens. Each search also embeds its query, a few tokens at the embedding price,
too small to change these totals.

In [19]:
PRICE_IN = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUT = float(settings["PRICE_OUTPUT_PER_MILLION"])


def cost(calls):
    tokens_in = sum(u.prompt_tokens for u in calls)
    tokens_out = sum(u.completion_tokens for u in calls)
    return len(calls), tokens_in, tokens_out, (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1_000_000


runs = {
    "single search, k=2": [single_k2.usage],
    "single search, k=6": [single_k6.usage],
    "agent": usage,
    "single, k=6, reworded": [single_k6_reworded.usage],
    "agent, reworded": usage_reworded,
}
print(f"{'run':24}{'calls':>6}{'input':>8}{'output':>8}{'US dollars':>12}")
for label, calls in runs.items():
    n, tokens_in, tokens_out, dollars = cost(calls)
    print(f"{label:24}{n:>6}{tokens_in:>8}{tokens_out:>8}{dollars:>12.6f}")

run                      calls   input  output  US dollars
single search, k=2           1     224      50    0.000047
single search, k=6           1     555      41    0.000076
agent                        3    1196      94    0.000167
single, k=6, reworded        1     528      40    0.000073
agent, reworded              4    1980     143    0.000269


On the rep's question the agent cost about 2.2 times the `k=6` search (0.000167 against 0.000076 US dollars). On the rewording, where `k=6` missed a fact, the agent cost 0.000269. All of them are a small fraction of a cent. These are single runs: the agent's number of turns, and so its cost, changes from run to run.

### In production

The search tool is the same idea in every framework. OpenAI's Responses API has a hosted `file_search` tool: the files
live in a vector store on OpenAI's side, and the model writes the queries while OpenAI runs the searches. LangChain
and LlamaIndex wrap a retriever as a tool you give to an agent. In each case it is worth printing the queries the model
writes, as this notebook did.

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `BadRequestError: Error code: 400 - Function tools with reasoning_effort are not supported for <model> in /v1/chat/completions.` | on Chat Completions this model calls tools only when the reasoning effort is `none` | set `REASONING_EFFORT=none` in `models.env` and run the first cell again |
| `TypeError: 'NoneType' object is not subscriptable` in Part B | the model answered instead of asking for a search, so `reply.tool_calls` is `None` | print `reply.content` to read the answer, then run Part B again from *Turn 1* |

## Challenge

1. **Helena's rep and the first-reply time.** Use `run_loop` on: *Helena received the wrong album. Who is her
   assigned support representative, and how soon should support send a first reply?* Ticket 01
   (`ticket_01_helena_wrong_album.md`) names her rep, Steve Johnson; the support-hours policy
   (`policy_support_hours.md`) says the first reply comes within 1 business day. Print the queries and check that the
   answer has both facts. Expected: both facts, from both files.
2. **Was the agent needed?** Look at which files the first search returned, and where each later query's words came
   from (by eye, or with `borrowed_words` from Going further). Then run `answer_once` on the same question with `k=2`
   and print the two nearest files. Expected: the first search, written from the question's words, already returns
   the ticket and the support-hours policy; any later search adds no new fact, even if its query borrows words from
   those files. The two nearest files are the same two, and the single search answers both facts in one call. Say why
   this question did not need a second search when the warped-vinyl question did.

The worked solution, run with its outputs, is in `solved/13b_agentic_rag_at_the_desk_solution.ipynb`.

## Resources

- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): tool schemas,
  `tool_calls`, parallel tool calls, and sending results back
- [OpenAI: vector embeddings](https://developers.openai.com/api/docs/guides/embeddings): what an embedding is, and
  several texts embedded in one request
- [OpenAI: file search](https://developers.openai.com/api/docs/guides/tools-file-search): the hosted search tool on
  the Responses API, over a vector store OpenAI keeps

Next: [Lab 14: How to search your documents with LlamaIndex](../14_LlamaIndex_RAG/README.md). Back to
[the lab list](../README.md).